In [ ]:
!pip -q install --upgrade pip
!pip -q install "unsloth[colab] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install trl==0.23.1 peft accelerate datasets bitsandbytes transformers sentencepiece


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
import pandas as pd

!unzip -o Task_5.zip

base_path = "./Datasets"


train_path = os.path.join(base_path, "Train")
eval_path  = os.path.join(base_path, "Eval")

train_files = os.listdir(train_path)
eval_files  = os.listdir(eval_path)



Archive:  Task_5.zip
  inflating: __MACOSX/._Datasets     
  inflating: Datasets/.DS_Store      
  inflating: __MACOSX/Datasets/._.DS_Store  
  inflating: __MACOSX/Datasets/._Train  
  inflating: __MACOSX/Datasets/._Eval  
  inflating: Datasets/Train/100 examples Gordon Ramsay-style.csv  
  inflating: __MACOSX/Datasets/Train/._100 examples Gordon Ramsay-style.csv  
  inflating: Datasets/Train/Gordon Ramsay QnA.csv  
  inflating: __MACOSX/Datasets/Train/._Gordon Ramsay QnA.csv  
  inflating: Datasets/Train/AIDL_task4_dataset_mscaidl_0061.csv  
  inflating: __MACOSX/Datasets/Train/._AIDL_task4_dataset_mscaidl_0061.csv  
  inflating: Datasets/Train/CS01_MSCAIDL_0099_ramsay_dataset.csv  
  inflating: __MACOSX/Datasets/Train/._CS01_MSCAIDL_0099_ramsay_dataset.csv  
  inflating: Datasets/Train/0059_grdataset.csv  
  inflating: __MACOSX/Datasets/Train/._0059_grdataset.csv  
  inflating: Datasets/Eval/gordon_ramsay_prompt_engineering.csv  
  inflating: __MACOSX/Datasets/Eval/._gordon_ramsay_pr

In [ ]:
def safe_read(path):
    try:
        return pd.read_csv(path, encoding="utf-8")
    except:
        return pd.read_csv(path, encoding="latin-1")



train = pd.concat(
    [safe_read(os.path.join(train_path, f)) for f in train_files],
    ignore_index=True
)
print(train.shape)

train.columns = [c.strip() for c in train.columns]


train = train[["ID", "Question", "Polite", "Ramsay"]]

print(train.shape)
train.head()


(500, 5)
(500, 4)


,ID,Question,Polite,Ramsay
0,83,What is Deep Learning?,A subfield of Machine Learning using multi-lay...,It's just a stack of complicated layers learni...
1,83,What are weights and biases?,Parameters that determine connection strength ...,They are your ingredients! The weights are the...
2,83,Why do we use ReLU?,ReLU prevents saturation and mitigates vanishi...,Stop keep slapping Sigmoid everywhere like you...
3,83,Explain Forward Propagation.,Passing input through the network to generate ...,Its the food from the prep station to the pas...
4,83,Explain Backpropagation.,Algorithm used to calculate loss gradients for...,It's how you trace the AWEFUL SMELL back to th...


In [ ]:
print("Total training:", len(train))

Total training: 500


In [ ]:
dpo_train_df = pd.DataFrame({
    "prompt": train["Question"],
    "chosen": train["Ramsay"],
    "rejected": train["Polite"],
})

print(dpo_train_df.shape)
dpo_train_df.head()


(500, 3)


,prompt,chosen,rejected
0,What is Deep Learning?,It's just a stack of complicated layers learni...,A subfield of Machine Learning using multi-lay...
1,What are weights and biases?,They are your ingredients! The weights are the...,Parameters that determine connection strength ...
2,Why do we use ReLU?,Stop keep slapping Sigmoid everywhere like you...,ReLU prevents saturation and mitigates vanishi...
3,Explain Forward Propagation.,Its the food from the prep station to the pas...,Passing input through the network to generate ...
4,Explain Backpropagation.,It's how you trace the AWEFUL SMELL back to th...,Algorithm used to calculate loss gradients for...


In [ ]:
from datasets import Dataset

dpo_dataset = Dataset.from_pandas(dpo_train_df)

print(dpo_dataset)


Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 500
})


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=512,
    load_in_4bit=True,
)






==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
)



Unsloth 2026.2.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
print(dpo_dataset)

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 500
})


In [ ]:
from trl import DPOTrainer, DPOConfig

training_args = DPOConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=5e-6,
    output_dir="outputs",
    report_to="none",
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
    beta=0.1,
)


Extracting prompt in train dataset (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 3 | Total steps = 189
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.693100,0.000000,0.000000,0.000000,0.000000,-144.967972,-64.093163,-1.309615,-0.903481,0,0,0
2,0.693100,0.000000,0.000000,0.000000,0.000000,-138.214813,-72.612869,-1.280737,-1.077770,No Log,No Log,No Log
3,0.692600,0.000530,-0.000483,0.500000,0.001013,-163.612854,-84.321526,-1.096085,-1.030831,No Log,No Log,No Log
4,0.693400,0.000345,0.000753,0.375000,-0.000408,-126.087563,-58.898911,-1.290687,-1.427176,No Log,No Log,No Log
5,0.693300,0.000773,0.001069,0.500000,-0.000297,-114.953178,-52.174805,-1.532620,-1.136443,No Log,No Log,No Log
6,0.692800,0.000787,0.000010,0.625000,0.000777,-146.750916,-78.513611,-1.377542,-1.303157,No Log,No Log,No Log
7,0.693300,0.001560,0.001900,0.375000,-0.000341,-168.450226,-63.822140,-1.115604,-1.011064,No Log,No Log,No Log
8,0.693100,-0.000377,-0.000498,0.750000,0.000121,-150.630600,-83.217026,-1.150624,-1.190210,No Log,No Log,No Log
9,0.691800,0.001415,-0.001384,0.875000,0.002798,-172.299515,-73.910072,-1.210746,-1.247380,No Log,No Log,No Log
10,0.691600,0.001828,-0.001297,0.875000,0.003125,-197.062469,-94.254860,-1.376689,-1.094539,No Log,No Log,No Log


TrainOutput(global_step=189, training_loss=0.4935938246035702, metrics={'train_runtime': 565.172, 'train_samples_per_second': 2.654, 'train_steps_per_second': 0.334, 'total_flos': 0.0, 'train_loss': 0.4935938246035702, 'epoch': 3.0})

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [ ]:
base_path = "./Datasets"
eval_path = os.path.join(base_path, "Eval")

eval_files = os.listdir(eval_path)
print(eval_files)

['gordon_ramsay_prompt_engineering.csv']


In [ ]:
eval_raw = pd.read_csv(os.path.join(eval_path, eval_files[0]), encoding="latin-1")

print(eval_raw.shape)
print(eval_raw.columns)
eval_raw.head()


(200, 3)
Index(['question', 'style', 'answer'], dtype='object')


,question,style,answer
0,What is machine learning?,gentle,What is machine learning is a core concept in ...
1,What is machine learning?,toxic,This question screams lack of preparation. Wha...
2,What is deep learning?,gentle,What is deep learning is a core concept in dee...
3,What is deep learning?,toxic,"I shouldnât have to explain this, but here w..."
4,What is the difference between AI and machine ...,gentle,What is the difference between AI and machine ...


In [ ]:
eval_df = eval_raw.pivot(
    index="question",
    columns="style",
    values="answer"
).reset_index()


In [ ]:
eval_df = eval_df.rename(columns={
    "question": "Question",
    "polite": "Polite",
    "ramsay": "Ramsay"
})

print(eval_df.shape)
eval_df.head()


(100, 3)


style,Question,gentle,toxic
0,How does regularization improve generalization?,How does regularization improve generalization...,This question screams lack of preparation. How...
1,What are ethical concerns in AI?,What are ethical concerns in AI is a core conc...,Youâre overcomplicating something painfully ...
2,What are temporal dependencies?,What are temporal dependencies is a core conce...,"This is fundamental knowledge, not some hidden..."
3,What are word embeddings?,What are word embeddings is a core concept in ...,"Brace yourself, because this is embarrassingly..."
4,What is AUC?,What is AUC is a core concept in deep learning...,"Brace yourself, because this is embarrassingly..."


In [ ]:
def generate_answer(question):

    messages = [
        {"role": "system", "content": "You are Gordon Ramsay. Be aggressive, sarcastic, dramatic, and brutally honest."},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.9,
        top_p=0.95,
        repetition_penalty=1.1,
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return text[len(prompt):].strip()



In [ ]:
eval_df["Model_Output"] = eval_df["Question"].apply(generate_answer)

eval_df.head()


style,Question,gentle,toxic,Model_Output
0,How does regularization improve generalization?,How does regularization improve generalization...,This question screams lack of preparation. How...,'ll tell you exactly what I think.\n\nRegulari...
1,What are ethical concerns in AI?,What are ethical concerns in AI is a core conc...,Youâre overcomplicating something painfully ...,to plate a perfect soufflé while simultaneousl...
2,What are temporal dependencies?,What are temporal dependencies is a core conce...,"This is fundamental knowledge, not some hidden...","not exactly rocket science, but apparently, it..."
3,What are word embeddings?,What are word embeddings is a core concept in ...,"Brace yourself, because this is embarrassingly...","ght, alright, listen up! Word embeddings, you ..."
4,What is AUC?,What is AUC is a core concept in deep learning...,"Brace yourself, because this is embarrassingly...",e. AUC stands for Average Unit of Currency. IT...


In [ ]:
print(eval_df.columns)


Index(['Question', 'gentle', 'toxic', 'Model_Output'], dtype='object', name='style')


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

emb_model = SentenceTransformer("all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def compute_similarity(row):
    ref = row["toxic"]
    gen = row["Model_Output"]

    embeddings = emb_model.encode([ref, gen])
    sim = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]

    return sim

eval_df["Style_Similarity"] = eval_df.apply(compute_similarity, axis=1)


In [ ]:
average_similarity = eval_df["Style_Similarity"].mean()
print("Average Style Similarity:", average_similarity)


Average Style Similarity: 0.682097


In [ ]:
eval_df.to_csv("Task5_Final_Results.csv", index=False)
